# 00 — Setup and sanity check

Run this first in any new Colab session to confirm the data loaded correctly before doing real work in another notebook. Every notebook in this repo starts with the same bootstrap cell below.

In [ ]:
import os
import sys

from google.colab import userdata

token = userdata.get("GH_PAT")  # set once via the key icon in Colab's left sidebar

BRANCH = "issue-1-meds-reader-wrapper"  # change this back to "main" once PR #21 is merged

REPO_DIR = "ehrshot-patient-graphs"
if os.path.exists(REPO_DIR):
    # Re-running without a fresh VM (no restart) — make sure we're actually
    # on BRANCH's latest commit instead of silently keeping a stale clone.
    os.chdir(REPO_DIR)
    !git fetch origin
    !git checkout {BRANCH}
    !git reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} https://{token}@github.com/patient-graph-lab/ehrshot-patient-graphs.git
    os.chdir(REPO_DIR)

!pip install -e ".[dev,data]"

# pip install -e . registers the package via a .pth file, which Python only
# reads at interpreter startup. Reinstalling into an already-running kernel
# (e.g. rerunning this cell without restarting the runtime) succeeds at the
# filesystem level -- "Successfully installed ehrgraph-0.1.0" -- but
# "import ehrgraph" still fails until the kernel restarts, because the
# running interpreter never re-scans for new sys.path entries. Adding src/
# directly sidesteps that: takes effect immediately, no restart needed,
# whether this is a fresh VM or a rerun.
src_path = os.path.join(os.getcwd(), "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

## STOP — manual step before running the next cell

**Update, Sep 20: Stanford's data stewards confirmed Google Drive is not an acceptable storage location for this data** (general Drive doesn't meet their security requirements — only institution-managed Drive setups with documented protocols would qualify, which this isn't). The Drive pass-through has been removed as an option, not just demoted. If you still have a standing copy in `ehrshot_transfer` from before this, delete it now and empty Drive's Trash — see the team channel for the cleanup snippet.

The next cell prompts you to upload the dataset directly — no Drive, no third-party storage of any kind. It does **not** prepare the files for you; that happens on your own machine first, before you run the cell below.

Get the script (this repo is private, so a plain `curl` of the raw file won't work without auth — reuse the same GitHub token you generated for Colab's `GH_PAT` secret):

```bash
export GH_PAT=<paste your token here, this one time>
git clone --branch issue-1-meds-reader-wrapper https://$GH_PAT@github.com/patient-graph-lab/ehrshot-patient-graphs.git
```

(Swap `issue-1-meds-reader-wrapper` for `main` once PR #21 is merged.)

Then, from the same folder as your 4 dataset files:

```bash
python /path/to/ehrshot-patient-graphs/scripts/dismantle_data.py EHRSHOT_ASSETS.zip
```

This creates one folder, `./data-chunks/`, containing everything you need to upload: `EHRSHOT_ASSETS.zip` split into pieces (the only one of the 4 files too large to upload directly), plus copies of the other 3 files. The original 4.3GB file is left alone outside that folder, on purpose — so there's nothing sitting there to accidentally re-select.

1. Run the commands above.
2. Run the next cell — it'll pop up an upload dialog.
3. Open `data-chunks/` and select **everything in it** — that's the whole list, nothing outside that folder is needed.
4. The pieces get reassembled into `EHRSHOT_ASSETS.zip` automatically.

**Note: this is slower than Drive was** (measured hours, not minutes, per session — see `DATA_ACCESS.md`). Stanford's suggested alternative — running the notebooks natively within Redivis instead of Colab — is being evaluated as a better long-term fix for this; see `DATA_ACCESS.md` for status.

In [ ]:
from scripts.colab_setup import ensure_ehrshot_data

data_root = ensure_ehrshot_data(checksums_path="data/checksums.txt")

In [ ]:
# Sanity check: confirm the extracted assets look right before doing anything else

assets_dir = data_root / "EHRSHOT_ASSETS"
for sub in ["benchmark", "features", "models", "results", "splits", "data"]:
    p = assets_dir / sub
    print(f"{sub:12s} exists={p.exists()}")

In [ ]:
import pandas as pd

labels = pd.read_csv(assets_dir / "benchmark" / "all_labels.csv")
print(labels.shape)
labels.head()

In [ ]:
import ehrgraph

print("ehrgraph package imports cleanly:", ehrgraph.__name__)

## Verify meds_reader against the real data (issue #1)

This is the one check that can't be done outside Colab — it needs the real, extracted MEDS reader database. Run the cell below and confirm it prints real events with no errors.

In [ ]:
from ehrgraph.data.loader import open_database

db_path = data_root / "meds_reader_omop_ehrshot"
db = open_database(str(db_path))

first_subject_id = next(iter(db))
subject = db[first_subject_id]
print(f"subject_id={subject.subject_id}")
for i, event in enumerate(subject.events):
    if i >= 5:
        break
    print(f"  {event.time}  {event.code}")